In [5]:
import sys
import os
import torch
import numpy as np

# Add the project root directory to sys.path to enable imports from the src folder
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.wrappers.hf_text import HFTextWrapper

# Determine the computation device (CUDA for NVIDIA, MPS for Mac, CPU otherwise)
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Environment Setup Complete.")
print(f"Project Root: {project_root}")
print(f"Active Device: {device.upper()}")

Environment Setup Complete.
Project Root: e:\03_Coding\xai-framework-lorenzo-gatta
Active Device: CPU


In [23]:
# --- TEST CONFIGURATION: CLASSIFICATION ---
model_id = "distilbert-base-uncased-finetuned-sst-2-english"
input_text = "Even though the movie theme was good, the plot was predictable and boring."

print(f"--- STARTING TEST: {model_id} ---")

# 1. Initialize the wrapper
# This step loads the model configuration, tokenizer, and weights.
print("Initializing model wrapper...")
wrapper = HFTextWrapper(model_id, device)

# 2. Perform Inference
# The generate method returns raw logits and tokenized inputs.
print(f"Processing input: '{input_text}'")
output = wrapper.generate(input_text)

# 3. Analyze Logits
logits = output["logits"]
print(f"Logits Shape: {logits.shape}")  # Expected: [1, 2]

# Convert logits to probabilities using Softmax for interpretation
probs = torch.nn.functional.softmax(logits, dim=-1)
predicted_index = torch.argmax(logits, dim=-1).item()

# Define labels for the SST-2 dataset
labels_map = {0: "NEGATIVE", 1: "POSITIVE"}

print(f"Raw Logits: {logits.detach().cpu().numpy()}")
print(f"Probabilities: {probs.detach().cpu().numpy()}")
print(f"Predicted Class Index: {predicted_index}")
print(f"Predicted Label: {labels_map.get(predicted_index, 'UNKNOWN')}")

# 4. Verify Tokenization
# Ensure tokens are correctly mapped to strings
print(f"Tokens: {output['tokens']}")

# 5. Verify Embedding Layer Retrieval
# This checks if the wrapper allows access to embeddings (required for Captum).
try:
    emb_layer = wrapper.get_embedding_layer()
    print(f"Embedding Layer Found: {type(emb_layer).__name__}")
except ValueError as e:
    print(f"Error: Could not find embedding layer. Details: {e}")

print("--- TEST COMPLETED ---")

--- STARTING TEST: distilbert-base-uncased-finetuned-sst-2-english ---
Initializing model wrapper...
Processing input: 'Even though the movie theme was good, the plot was predictable and boring.'
Logits Shape: torch.Size([1, 2])
Raw Logits: [[ 4.654342  -3.7759547]]
Probabilities: [[9.9978191e-01 2.1810913e-04]]
Predicted Class Index: 0
Predicted Label: NEGATIVE
Tokens: ['[CLS]', 'even', 'though', 'the', 'movie', 'theme', 'was', 'good', ',', 'the', 'plot', 'was', 'predictable', 'and', 'boring', '.', '[SEP]']
Embedding Layer Found: Embeddings
--- TEST COMPLETED ---


In [24]:
# --- TEST CONFIGURATION: GENERATION ---
model_id = "gpt2"
input_text = "The NASA is"

print(f"--- STARTING TEST: {model_id} ---")

# 1. Initialize the wrapper
print("Initializing model wrapper...")
wrapper = HFTextWrapper(model_id, device)

# 2. Perform Inference
print(f"Processing input: '{input_text}'")
output = wrapper.generate(input_text)

# 3. Analyze Logits
logits = output["logits"]
print(f"Logits Shape: {logits.shape}") 
# Expected: [1, Sequence Length, Vocab Size]

# Check consistency with tokenizer vocabulary
vocab_size = wrapper.tokenizer.vocab_size
print(f"Tokenizer Vocabulary Size: {vocab_size}")

if logits.shape[-1] == vocab_size:
    print("Verification Passed: Logits dimension matches vocabulary size.")
else:
    print(f"Note: Logits dimension ({logits.shape[-1]}) differs from standard vocab size.")

# 4. Verify Tokenization
# Observe GPT-2 specific artifacts (like the 'Ġ' character for spaces)
print(f"Tokens: {output['tokens']}")

# 5. Verify Embedding Layer Retrieval
try:
    emb_layer = wrapper.get_embedding_layer()
    print(f"Embedding Layer Found: {type(emb_layer).__name__}")
except ValueError as e:
    print(f"Error: Could not find embedding layer. Details: {e}")

# Optional: Print other available keys in the output dictionary
print(f"Available Output Keys: {list(output.keys())}")

print("--- TEST COMPLETED ---")

--- STARTING TEST: gpt2 ---
Initializing model wrapper...
Processing input: 'The NASA is'
Logits Shape: torch.Size([1, 3, 50257])
Tokenizer Vocabulary Size: 50257
Verification Passed: Logits dimension matches vocabulary size.
Tokens: ['The', 'ĠNASA', 'Ġis']
Embedding Layer Found: Embedding
Available Output Keys: ['logits', 'input_ids', 'attention_mask', 'tokens']
--- TEST COMPLETED ---


In [25]:
# --- EXTENSION: CHECKING GENERATION ---

print("\n--- GENERATION CHECK ---")

# 1. Predict the NEXT single token based on current logits
# We look at the logits corresponding to the LAST token of the input sequence
# Shape: [1, Sequence_Length, Vocab_Size] -> take the last step of sequence
last_token_logits = logits[0, -1, :] 
predicted_id = torch.argmax(last_token_logits).item()
predicted_word = wrapper.tokenizer.decode(predicted_id)

print(f"Input: '{input_text}'")
print(f"Most likely NEXT word: '{predicted_word}'")

# 2. Generate a full sentence (using the underlying HF model directly)
# We access wrapper.model directly to use the native generate() method for text completion
print("Generating full continuation...")
inputs = wrapper.tokenizer(input_text, return_tensors="pt").to(device)

# Generate up to 10 new tokens
generated_ids = wrapper.model.generate(
    **inputs, 
    max_new_tokens=10, 
    pad_token_id=wrapper.tokenizer.eos_token_id
)

full_text = wrapper.tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(f"Full generated text: '{full_text}'")


--- GENERATION CHECK ---
Input: 'The NASA is'
Most likely NEXT word: ' working'
Generating full continuation...
Full generated text: 'The NASA is working on a new system that will allow astronauts to'
